In [1]:
import sys
sys.path.append('..')
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model
from utils.token_utils import count_text_tokens

In [4]:
with open('..\\data\\Incidents.txt', 'r') as file:
    lines = file.readlines()

incidents = lines[1:]

token_limit = 150

model = pick_model('openai', 'general')
# Use hard_prompt_cap to trigger truncation
client_capped = LLMClient('openai', model, hard_prompt_cap=300)

print(f'Using reasoning model: {model}')

for i in range(len(incidents)):
    cols = [c.strip() for c in incidents[i].split('|')]
    # message = cols[-1]*5000
    message = cols[-1]

    text_token_count = count_text_tokens(
        message,
        provider='openai',
        model=model
    )
    
    print(f'Incident {i+1} token count: {text_token_count}')
    
    if text_token_count > token_limit:    
        prompt_text, spec = render(
            'overflow_summarize.v1',
            context=message, 
            max_tokens_context='150',
            task='Summarize the message briefly',
            format='Single paragraph'
        )
        
        messages = [{'role': 'user', 'content': prompt_text}]

        response = client_capped.chat(messages, temperature=0.2)
        print('Overflow handled:', response['meta']['overflow_handled'])
        print("BLOCKED/TRUNCATED")
        print(response['text'])

        log_llm_call('openai', model, 'overflow_summarize', response['latency_ms'], response['usage'])

    else:
        print("Message within token budget.")

Using reasoning model: gpt-4o-mini
Incident 1 token count: 13
Message within token budget.
Incident 2 token count: 11
Message within token budget.
Incident 3 token count: 10
Message within token budget.
